<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/ocr_correction_transcripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspellchecker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 63.7 MB/s eta 0:00:00


In [ ]:
!pip install wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 10.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import sys
import re
import os
import re

import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
import pickle
import json

from pathlib import Path
import spacy
from spellchecker import SpellChecker

from multiprocessing import Pool
from multiprocessing import cpu_count
import math
import time
import random
from wordfreq import zipf_frequency

from collections import Counter
from pathlib import Path
import re

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
from nltk.corpus import words
nltk.download('words')


[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [ ]:
  >>> import nltk
  >>> nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


Mounted at /content/drive/


In [ ]:
mens_transcripts_full_path = Path('/content/drive/MyDrive/Capital_trials/men')
womens_transcripts_full_path = Path('/content/drive/MyDrive/Capital_trials/women')
corrected_mens_transcripts_path = Path('/content/drive/MyDrive/Capital_trials/men/corrected_docs')

In [ ]:
#pipeline to chunk docs run ocr correction, and restitch
TOKEN_RE = re.compile(r"[A-Za-z']+|[^\w\s]")

def tokenize(text: str):
    return TOKEN_RE.findall(text)

def detokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return text

def chunk_tokens(tokens, max_len=256, overlap=32):
    if max_len <= overlap:
        raise ValueError("max_len must be > overlap")
    step = max_len - overlap
    for start in range(0, len(tokens), step):
        end = start + max_len
        yield tokens[start:end]
        if end >= len(tokens):
            break

def chunk_doc_to_texts(text, max_len=256, overlap=32):
    tokens = tokenize(text)
    return [detokenize(ch) for ch in chunk_tokens(tokens, max_len, overlap)]

def correct_texts(texts, tok, model, batch_size=64, max_len=256, max_new_tokens=128, pbar=None):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tok(batch, return_tensors="pt", padding="max_length",
                     max_length=max_len, truncation=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outs = model.generate(**inputs, max_new_tokens=max_new_tokens)
        out.extend(tok.batch_decode(outs, skip_special_tokens=True))
        if pbar is not None:
            pbar.update(len(batch))
    return out

def remerge_corrected_chunks(corrected_chunks, overlap=32):
    if not corrected_chunks:
        return ""
    merged_tokens = []
    for i, chunk_text in enumerate(corrected_chunks):
        toks = tokenize(chunk_text)
        if i == 0:
            merged_tokens.extend(toks)
        else:
            merged_tokens.extend(toks[overlap:])  # drop overlap
    return detokenize(merged_tokens)

def correct_folder_end_to_end(
    input_dir,
    output_dir,
    model_id="pykale/bart-base-ocr",
    batch_size=64,
    max_len=256,
    max_new_tokens=128,
    overlap=32,
    recursive=True,
    save_chunks=False,
    raw_chunk_dir=None,
    corrected_chunk_dir=None,
):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to("cuda").eval()

    pattern = "**/*.txt" if recursive else "*.txt"
    files = sorted(input_dir.glob(pattern))

    for p in tqdm(files, desc="Files", unit="file"):
        rel = p.relative_to(input_dir)
        out_path = output_dir / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)

        text = p.read_text(encoding="utf-8", errors="ignore")
        chunks = chunk_doc_to_texts(text, max_len=max_len, overlap=overlap)

        if save_chunks and raw_chunk_dir:
            raw_base = Path(raw_chunk_dir) / rel.parent
            raw_base.mkdir(parents=True, exist_ok=True)
            for i, ch in enumerate(chunks, 1):
                (raw_base / f"{p.stem}_part_{i:06d}.txt").write_text(ch, encoding="utf-8")

        with tqdm(total=len(chunks), desc=f"Chunks: {p.name}", unit="chunk", leave=False) as pbar:
            corrected_chunks = correct_texts(
                chunks, tok, model,
                batch_size=batch_size,
                max_len=max_len,
                max_new_tokens=max_new_tokens,
                pbar=pbar
            )

        if save_chunks and corrected_chunk_dir:
            corr_base = Path(corrected_chunk_dir) / rel.parent
            corr_base.mkdir(parents=True, exist_ok=True)
            for i, ch in enumerate(corrected_chunks, 1):
                (corr_base / f"{p.stem}_part_{i:06d}.txt").write_text(ch, encoding="utf-8")

        corrected_text = remerge_corrected_chunks(corrected_chunks, overlap=overlap)
        out_path.write_text(corrected_text, encoding="utf-8")

In [ ]:
correct_folder_end_to_end(
    input_dir='/content/drive/MyDrive/Capital_trials/women',
    output_dir='/content/drive/MyDrive/Capital_trials/women' +"/" +"corrected_docs",
    batch_size=64,
    max_len=256,
    max_new_tokens=128,
    overlap=32,
    recursive=True,
    save_chunks=True,
    raw_chunk_dir="/content/drive/MyDrive/Capital_trials/womens_raw_chunks",
    corrected_chunk_dir="content/drive/MyDrive/Capital_trials/womens_corrected_chunks",
)


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Chunks: Antoinette Frank.txt:   0%|          | 0/361 [00:00<?, ?chunk/s]

In [ ]:
correct_folder_end_to_end(
    input_dir=mens_transcripts_full_path,
    output_dir='/content/drive/MyDrive/Capital_trials/men' +"/" +"corrected_docs",
    batch_size=64,
    max_len=256,
    max_new_tokens=128,
    overlap=32,
    recursive=True,
    save_chunks=True,
    raw_chunk_dir="/content/drive/MyDrive/Capital_trials/mens_raw_chunks",
    corrected_chunk_dir="content/drive/MyDrive/Capital_trials/mens_corrected_chunks",
)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/292 [00:00<?, ?B/s]

Files:   3%|▎         | 2/69 [12:14<6:08:25, 329.93s/file] 
Chunks: Andrew Lackey Combined TranscriptOCR.txt:   0%|          | 0/851 [00:00<?, ?chunk/s]
Chunks: Andrew Lackey Combined TranscriptOCR.txt:   8%|▊         | 64/851 [00:10<02:03,  6.37chunk/s]
Chunks: Andrew Lackey Combined TranscriptOCR.txt:  15%|█▌        | 128/851 [00:19<01:52,  6.45chunk/s]
Chunks: Andrew Lackey Combined TranscriptOCR.txt:  23%|██▎       | 192/851 [00:29<01:43,  6.40chunk/s]
Chunks: Andrew Lackey Combined TranscriptOCR.txt:  30%|███       | 256/851 [00:40<01:34,  6.31chunk/s]
Chunks: Andrew Lackey Combined TranscriptOCR.txt:  38%|███▊      | 320/851 [00:50<01:23,  6.34chunk/s]
Chunks: Andrew Lackey Combined TranscriptOCR.txt:  45%|████▌     | 384/851 [01:00<01:13,  6.36chunk/s]
Chunks: Andrew Lackey Combined TranscriptOCR.txt:  53%|█████▎    | 448/851 [01:10<01:03,  6.36chunk/s]
Chunks: Andrew Lackey Combined TranscriptOCR.txt:  60%|██████    | 512/851 [01:20<00:53,  6.38chunk/s]
Chunks: Andrew Lackey Co

KeyboardInterrupt: 